# Task 3: Granulometry Benchmarking â€” Qwen2.5-VL-3B (Base Model)

Benchmark the un-fine-tuned model on 108 test images of concrete aggregate.
Classify each image by max particle size (8/16/32mm) and grading (coarse/medium/fine).

Two modes:
- **Zero-shot** (1500px): No reference image
- **Few-shot** (ref@800 + test@1400): Reference classification chart included

This establishes the baseline that Task 4 (LoRA fine-tuning) should improve.

In [ ]:
import os, json, re, time, torch, gc
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} â€” {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

## Config & Prompts

In [ ]:
TEST_DIR = '../../datasets/granulometry/test'
MANIFEST = '../../datasets/granulometry/test_manifest.json'
REF_IMAGE_PATH = 'examples_classification_data.png'
ORIGINAL_GSD = 8.0
MAX_DIM_ZS = 1500   # zero-shot: single image
MAX_DIM_FS = 1400   # few-shot: test image (ref is kept at native ~600x445)
IMG_SIZE = 500       # preview thumbnails only

PROMPT_ZERO_SHOT = '''This is a top-down photo of concrete aggregate. GSD = {gsd:.1f} px/mm.

Classify it on two axes:

MAX PARTICLE SIZE - estimate the largest stone's width in pixels, divide by {gsd:.1f} to get mm, round to 8, 16, or 32.
Reference: at {gsd:.1f} px/mm, a 8mm stone = ~{eight:.0f}px wide, 16mm = ~{sixteen:.0f}px, 32mm = ~{thirtytwo:.0f}px.

GRADING - describes the size distribution relative to the max size:
- COARSE: most particles are similar in size, close to the max. Few small particles. Looks uniform.
- MEDIUM: moderate mix of large and small.
- FINE: wide range of sizes. Many small particles fill gaps between larger ones. Looks dense and packed.

Example: "coarse 32mm" = mostly 16-32mm stones, few small ones. "fine 32mm" = some 32mm stones but lots of 4-16mm particles filling every gap.

Respond with ONLY a JSON object (no other text):
{{"max_particle_size_mm": <8, 16, or 32>, "grading": "<coarse, medium, or fine>"}}'''

PROMPT_FEW_SHOT_REF = '''Reference chart: 3x3 grid of concrete aggregate photos.

COLUMNS (left to right) = max particle size: 8mm | 16mm | 32mm
ROWS (top to bottom) = grading: A (coarse) | B (medium) | C (fine)

What grading means - it describes size DISTRIBUTION, not absolute size:
- COARSE (A): particles are mostly the same size, close to the max. Few small ones. Uniform texture.
- MEDIUM (B): moderate mix of large and small.
- FINE (C): many different sizes. Small particles fill all gaps between big ones. Dense, packed texture.

Look at column 32mm: row A has mostly big uniform stones, row C has big stones BUT also lots of small ones filling every gap. That is the difference.'''

PROMPT_FEW_SHOT_QUERY = '''Classify this photo. GSD = {gsd:.1f} px/mm.

Compare to the reference grid:
1. Which COLUMN? (8, 16, or 32mm) - match the largest stone size.
   Hint: at {gsd:.1f} px/mm, a 8mm stone = ~{eight:.0f}px wide, 16mm = ~{sixteen:.0f}px, 32mm = ~{thirtytwo:.0f}px.
2. Which ROW? (A=coarse, B=medium, C=fine)
   - Uniform texture, most stones similar size = coarse (A)
   - Dense packed texture, many small particles filling gaps between big ones = fine (C)
   - In between = medium (B)

Respond with ONLY a JSON object (no other text):
{{"max_particle_size_mm": <8, 16, or 32>, "grading": "<coarse, medium, or fine>"}}'''


## Load Dataset

In [ ]:
with open(MANIFEST) as f:
    manifest = json.load(f)

print(f'Total test images: {len(manifest)}')
cls_counts = Counter(e['class'] for e in manifest)
for cls in sorted(cls_counts):
    print(f'  {cls}: {cls_counts[cls]} images')

In [ ]:
# Load reference image â€” white background, keep at native size
ref_image = None
if os.path.exists(REF_IMAGE_PATH):
    raw_ref = Image.open(REF_IMAGE_PATH).convert('RGBA')
    white_bg = Image.new('RGBA', raw_ref.size, (255, 255, 255, 255))
    ref_image = Image.alpha_composite(white_bg, raw_ref).convert('RGB')
    ref_image.thumbnail((800, 800), Image.Resampling.LANCZOS)
    print(f'Reference image: {ref_image.size}')
    plt.figure(figsize=(8, 6))
    plt.imshow(ref_image); plt.title('Reference Chart'); plt.axis('off'); plt.show()
else:
    print(f'WARNING: {REF_IMAGE_PATH} not found')

## Load Model

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2.5-VL-3B-Instruct', torch_dtype=torch.bfloat16,
    device_map='auto', max_memory={0: '6GiB', 1: '15GiB'},
)
print(f'Model loaded.')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.1f} GB')

## Helper Functions

In [ ]:
def parse_response(raw):
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```\s*', '', raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    size_m = re.search(r'"max_particle_size_mm"\s*:\s*(\d+)', raw)
    grad_m = re.search(r'"grading"\s*:\s*"(\w+)"', raw)
    if size_m and grad_m:
        return {'max_particle_size_mm': int(size_m.group(1)), 'grading': grad_m.group(1)}
    return None

def infer(img_path, mode='zero-shot', ref_img=None):
    image = Image.open(img_path).convert('RGB')
    max_dim = MAX_DIM_FS if (mode == 'few-shot' and ref_img is not None) else MAX_DIM_ZS
    orig_max = max(image.size)
    scale = min(max_dim / orig_max, 1.0)
    if scale < 1.0:
        image = image.resize((int(image.width * scale), int(image.height * scale)), Image.Resampling.LANCZOS)
    actual_gsd = ORIGINAL_GSD * scale
    if mode == 'few-shot' and ref_img is not None:
        msgs = [{'role':'user','content':[
            {'type':'image','image':ref_img},
            {'type':'text','text':PROMPT_FEW_SHOT_REF},
            {'type':'image','image':image},
            {'type':'text','text':PROMPT_FEW_SHOT_QUERY.format(gsd=actual_gsd, eight=8*actual_gsd, sixteen=16*actual_gsd, thirtytwo=32*actual_gsd)},
        ]}]
        images = [ref_img, image]
    else:
        msgs = [{'role':'user','content':[
            {'type':'image','image':image},
            {'type':'text','text':PROMPT_ZERO_SHOT.format(gsd=actual_gsd, eight=8*actual_gsd, sixteen=16*actual_gsd, thirtytwo=32*actual_gsd)},
        ]}]
        images = [image]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=images, return_tensors='pt', padding=True).to(model.device)
    t = time.time()
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    elapsed = time.time() - t
    out = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    del inputs, ids; image.close(); torch.cuda.empty_cache()
    return out.strip(), elapsed

def run_benchmark(manifest, mode, ref_img=None):
    results = []
    correct_size = 0; correct_grading = 0; valid_json = 0; total_time = 0
    for i, entry in enumerate(manifest):
        img_path = os.path.join(TEST_DIR, entry['image'])
        if not os.path.exists(img_path): continue
        raw, elapsed = infer(img_path, mode=mode, ref_img=ref_img)
        total_time += elapsed
        parsed = parse_response(raw)
        gt_size = entry['max_particle_size_mm']; gt_grading = entry['grading']
        size_ok = False; grading_ok = False; is_valid = parsed is not None
        if is_valid:
            valid_json += 1
            pred_size = parsed.get('max_particle_size_mm')
            if isinstance(pred_size, str): pred_size = int(pred_size) if pred_size.isdigit() else None
            if pred_size == gt_size: size_ok = True; correct_size += 1
            if parsed.get('grading', '').lower() == gt_grading: grading_ok = True; correct_grading += 1
        results.append({
            'image': entry['image'], 'class': entry['class'],
            'gt_size': gt_size, 'gt_grading': gt_grading,
            'predicted': parsed, 'raw': raw,
            'size_correct': size_ok, 'grading_correct': grading_ok,
            'valid_json': is_valid, 'time_s': round(elapsed, 2),
        })
        if (i+1) % 10 == 0:
            n = i + 1
            print(f'[{n}/{len(manifest)}] Size: {correct_size}/{n} ({correct_size/n*100:.0f}%) | '
                  f'Grading: {correct_grading}/{n} ({correct_grading/n*100:.0f}%) | '
                  f'JSON: {valid_json}/{n} | Avg: {total_time/n:.1f}s')
    return results, correct_size, correct_grading, valid_json, total_time

print('Helpers ready.')


## Quick Test (hidden)
Expand to run a sanity check — 1 image per class, zero-shot only.

In [ ]:
# Quick test: 1 image per class, zero-shot
all_classes = sorted(set(e['class'] for e in manifest))
quick_test = [next(e for e in manifest if e['class'] == cls) for cls in all_classes]
print(f'Quick test: {len(quick_test)} images — {[e["class"] for e in quick_test]}')

for entry in quick_test:
    img_path = os.path.join(TEST_DIR, entry['image'])
    raw, elapsed = infer(img_path, mode='zero-shot')
    parsed = parse_response(raw)
    gt_size = entry['max_particle_size_mm']; gt_grading = entry['grading']
    if parsed:
        ps = parsed.get('max_particle_size_mm'); pg = parsed.get('grading','').lower()
        if isinstance(ps, str): ps = int(ps) if ps.isdigit() else None
        sv = '✓' if ps == gt_size else '✗'; gv = '✓' if pg == gt_grading else '✗'
        print(f"{entry['class']:>3} | GT: {gt_size}mm {gt_grading} | Pred: {ps}mm {pg} | Size {sv} Grading {gv} | {elapsed:.1f}s")
    else:
        print(f"{entry['class']:>3} | GT: {gt_size}mm {gt_grading} | PARSE FAIL | {elapsed:.1f}s | Raw: {raw[:80]}")


## Diagnostic: Natural Language Reasoning (hidden)
Expand to run zero-shot and few-shot diagnostics with chain-of-thought reasoning.

In [ ]:
# Zero-shot diagnostic: 4 extreme classes, natural language
DIAG_ZS = '''This is a top-down photo of concrete aggregate. GSD = {gsd:.1f} px/mm.

Classify it on two axes:
MAX PARTICLE SIZE — estimate the largest stone width in pixels, divide by {gsd:.1f} to get mm, round to 8, 16, or 32.
GRADING — describes size distribution relative to max size:
- COARSE: most particles similar size, close to max. Few small particles. Uniform texture.
- MEDIUM: moderate mix of large and small.
- FINE: wide range of sizes. Many small particles fill gaps between larger ones. Dense, packed texture.

Think step by step, then state: max size, grading, and your confidence.'''

for cls in ['A8', 'A32', 'C8', 'C32']:
    entry = next(e for e in manifest if e['class'] == cls)
    img_path = os.path.join(TEST_DIR, entry['image'])
    image = Image.open(img_path).convert('RGB')
    scale = min(MAX_DIM_ZS / max(image.size), 1.0)
    actual_gsd = ORIGINAL_GSD * scale
    if scale < 1.0:
        image = image.resize((int(image.width*scale), int(image.height*scale)), Image.Resampling.LANCZOS)
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':DIAG_ZS.format(gsd=actual_gsd)}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=500, temperature=0.7, do_sample=True)
    out = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    del inputs, ids; image.close(); torch.cuda.empty_cache()
    print(f'\n{"=" * 60}')
    print(f'{cls} — GT: {entry["max_particle_size_mm"]}mm, {entry["grading"]}')
    print(f'{"=" * 60}')
    print(out.strip())


In [ ]:
# Few-shot diagnostic: 4 extreme classes, natural language
DIAG_FS_REF = '''Reference chart: 3x3 grid of concrete aggregate photos.
COLUMNS (left to right) = max particle size: 8mm | 16mm | 32mm
ROWS (top to bottom) = grading: A (coarse) | B (medium) | C (fine)
What grading means — size DISTRIBUTION, not absolute size:
- COARSE (A): particles mostly same size, close to max. Few small ones. Uniform texture.
- MEDIUM (B): moderate mix of large and small.
- FINE (C): many different sizes. Small particles fill all gaps. Dense, packed texture.
Look at column 32mm: row A = big uniform stones, row C = big stones + lots of small ones filling gaps.'''

DIAG_FS_Q = '''Classify this photo. GSD = {gsd:.1f} px/mm.
Compare to the reference grid:
1. Which COLUMN? (8, 16, or 32mm)
2. Which ROW? (A=coarse, B=medium, C=fine)
3. Name the cell and explain why.
Think step by step.'''

for cls in ['A8', 'A32', 'C8', 'C32']:
    entry = next(e for e in manifest if e['class'] == cls)
    img_path = os.path.join(TEST_DIR, entry['image'])
    image = Image.open(img_path).convert('RGB')
    scale = min(MAX_DIM_FS / max(image.size), 1.0)
    actual_gsd = ORIGINAL_GSD * scale
    if scale < 1.0:
        image = image.resize((int(image.width*scale), int(image.height*scale)), Image.Resampling.LANCZOS)
    msgs = [{'role':'user','content':[
        {'type':'image','image':ref_image},{'type':'text','text':DIAG_FS_REF},
        {'type':'image','image':image},{'type':'text','text':DIAG_FS_Q.format(gsd=actual_gsd)},
    ]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[ref_image, image], return_tensors='pt', padding=True).to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=500, temperature=0.7, do_sample=True)
    out = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    del inputs, ids; image.close(); torch.cuda.empty_cache()
    print(f'\n{"=" * 60}')
    print(f'{cls} — GT: {entry["max_particle_size_mm"]}mm, {entry["grading"]}')
    print(f'{"=" * 60}')
    print(out.strip())


## Full Benchmark — Zero-Shot
1500px, no reference image. ~10-15 min for 108 images.

In [ ]:
print('Running zero-shot benchmark on all', len(manifest), 'images...')
zs_results, zs_size, zs_grading, zs_json, zs_time = run_benchmark(manifest, 'zero-shot')
print(f'\nDone. {len(zs_results)} images in {zs_time:.0f}s')


## Full Benchmark — Few-Shot
ref@800 + test@1400. ~15-20 min for 108 images.

In [ ]:
print('Running few-shot benchmark on all', len(manifest), 'images...')
fs_results, fs_size, fs_grading, fs_json, fs_time = run_benchmark(manifest, 'few-shot', ref_img=ref_image)
print(f'\nDone. {len(fs_results)} images in {fs_time:.0f}s')


## Results Comparison

In [ ]:
def summarize(label, results, c_size, c_grading, v_json, t_time):
    n = len(results)
    both = sum(1 for r in results if r['size_correct'] and r['grading_correct'])
    return {'label': label, 'n': n,
        'json_pct': round(v_json/n*100, 1), 'size_pct': round(c_size/n*100, 1),
        'grading_pct': round(c_grading/n*100, 1), 'both_pct': round(both/n*100, 1),
        'avg_time': round(t_time/n, 2)}

zs_s = summarize('Zero-Shot', zs_results, zs_size, zs_grading, zs_json, zs_time)
fs_s = summarize('Few-Shot', fs_results, fs_size, fs_grading, fs_json, fs_time)

print(f'{"":<16} {"Zero-Shot":>12} {"Few-Shot":>12} {"Delta":>8}')
print('-' * 50)
for key, label in [('json_pct','JSON Valid %'),('size_pct','Size Acc %'),('grading_pct','Grading Acc %'),('both_pct','Both Correct %'),('avg_time','Avg Time (s)')]:
    zv = zs_s[key]; fv = fs_s[key]; d = fv - zv
    print(f'{label:<16} {zv:>12} {fv:>12} {("+" if d>0 else "")}{d:>7.1f}')


## Per-Class Breakdown

In [ ]:
def class_breakdown(results):
    by_class = defaultdict(list)
    for r in results: by_class[r['class']].append(r)
    data = {}
    for cls in sorted(by_class):
        cr = by_class[cls]
        data[cls] = {
            'size': sum(1 for r in cr if r['size_correct']) / len(cr) * 100,
            'grading': sum(1 for r in cr if r['grading_correct']) / len(cr) * 100,
            'both': sum(1 for r in cr if r['size_correct'] and r['grading_correct']) / len(cr) * 100,
        }
    return data

zs_cls = class_breakdown(zs_results)
fs_cls = class_breakdown(fs_results)
classes = sorted(zs_cls.keys())

print(f'{"Class":<6} {"--- Zero-Shot ---":>20} {"--- Few-Shot ---":>20}')
print(f'{"":<6} {"Size":>6} {"Grade":>6} {"Both":>6} {"Size":>6} {"Grade":>6} {"Both":>6}')
print('-' * 44)
for cls in classes:
    z = zs_cls[cls]; f = fs_cls[cls]
    print(f'{cls:<6} {z["size"]:>5.0f}% {z["grading"]:>5.0f}% {z["both"]:>5.0f}% {f["size"]:>5.0f}% {f["grading"]:>5.0f}% {f["both"]:>5.0f}%')


## Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(classes)); w = 0.35

# Size accuracy
axes[0].bar(x - w/2, [zs_cls[c]['size'] for c in classes], w, label='Zero-Shot', color='steelblue')
axes[0].bar(x + w/2, [fs_cls[c]['size'] for c in classes], w, label='Few-Shot', color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(classes, rotation=45)
axes[0].set_ylabel('Accuracy %'); axes[0].set_title('Size Accuracy by Class'); axes[0].legend(); axes[0].set_ylim(0, 110)

# Grading accuracy
axes[1].bar(x - w/2, [zs_cls[c]['grading'] for c in classes], w, label='Zero-Shot', color='steelblue')
axes[1].bar(x + w/2, [fs_cls[c]['grading'] for c in classes], w, label='Few-Shot', color='coral')
axes[1].set_xticks(x); axes[1].set_xticklabels(classes, rotation=45)
axes[1].set_ylabel('Accuracy %'); axes[1].set_title('Grading Accuracy by Class'); axes[1].legend(); axes[1].set_ylim(0, 110)

# Overall comparison
metrics = ['JSON Valid', 'Size Acc', 'Grading Acc', 'Both Correct']
zv = [zs_s['json_pct'], zs_s['size_pct'], zs_s['grading_pct'], zs_s['both_pct']]
fv = [fs_s['json_pct'], fs_s['size_pct'], fs_s['grading_pct'], fs_s['both_pct']]
mx = np.arange(len(metrics))
b1 = axes[2].bar(mx - w/2, zv, w, label='Zero-Shot', color='steelblue')
b2 = axes[2].bar(mx + w/2, fv, w, label='Few-Shot', color='coral')
axes[2].set_xticks(mx); axes[2].set_xticklabels(metrics); axes[2].set_ylim(0, 110)
axes[2].set_ylabel('Percentage'); axes[2].set_title('Overall Metrics'); axes[2].legend()
for bars in [b1, b2]:
    for bar in bars:
        axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{bar.get_height():.0f}%', ha='center', fontsize=9)

plt.suptitle('Task 3: Granulometry Benchmark — Qwen2.5-VL-3B (Base Model)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


## Save Results

In [ ]:
for label, summary, results in [
    ('zero-shot', zs_s, zs_results),
    ('few-shot', fs_s, fs_results),
]:
    fname = f'benchmark_results_{label}.json'
    with open(fname, 'w') as f:
        json.dump({
            'model': 'Qwen2.5-VL-3B-Instruct',
            'mode': label,
            'phase': 'base_model',
            'total_images': summary['n'],
            'json_validity_pct': summary['json_pct'],
            'size_accuracy_pct': summary['size_pct'],
            'grading_accuracy_pct': summary['grading_pct'],
            'both_correct_pct': summary['both_pct'],
            'avg_inference_time_s': summary['avg_time'],
            'results': results,
        }, f, indent=2)
    print(f'Saved {fname}')

print('\nBaseline established. Compare against Task 4 (LoRA fine-tuned) results.')
print(f'Progression: zero-shot → few-shot → fine-tuned')


## Frontier Model Benchmark

Before using a frontier model (Claude Opus 4.6 / GPT-5) to generate training data for Task 4,
verify it can actually classify these images correctly.

If the frontier model also fails, there is no point using it as a teacher.

### Frontier Config

In [ ]:
import base64
from openai import AzureOpenAI

# Azure OpenAI — GPT-5-mini on ether-project-resource
AZURE_OPENAI_ENDPOINT = 'https://ether-project-resource.openai.azure.com/'
AZURE_OPENAI_DEPLOYMENT = 'gpt-5-mini'
AZURE_OPENAI_API_VERSION = '2024-12-01-preview'
AZURE_OPENAI_API_KEY = os.environ.get('AZURE_OPENAI_API_KEY', '')  # Set via: export AZURE_OPENAI_API_KEY=your-key

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

FRONTIER_PROMPT_ZS = '''This is a top-down photo of concrete aggregate. GSD = 8.0 px/mm.

Classify it on two axes:

MAX PARTICLE SIZE - estimate the largest stone, round to 8, 16, or 32 mm.
Reference: at 8.0 px/mm, a 8mm stone = ~64px wide, 16mm = ~128px, 32mm = ~256px.

GRADING - describes size distribution (not absolute size):
- COARSE: most particles similar size, close to max. Few small particles. Uniform texture.
- MEDIUM: moderate mix of large and small.
- FINE: wide range of sizes. Many small particles fill gaps between larger ones. Dense, packed.

Respond with ONLY a JSON object:
{"max_particle_size_mm": <8|16|32>, "grading": "<coarse|medium|fine>"}'''

FRONTIER_PROMPT_FS_REF = '''First image: reference chart. 3x3 grid of concrete aggregate photos.
COLUMNS (left to right) = max particle size: 8mm | 16mm | 32mm
ROWS (top to bottom) = grading: A (coarse) | B (medium) | C (fine)

Grading = size DISTRIBUTION:
- COARSE (A): uniform texture, particles mostly same size near max.
- MEDIUM (B): moderate mix.
- FINE (C): dense packed texture, many small particles filling gaps between big ones.

Column 32mm: row A = big uniform stones, row C = big stones + lots of small ones filling gaps.'''

FRONTIER_PROMPT_FS_QUERY = '''Classify this photo. GSD = 8.0 px/mm.
Compare to the reference grid.
Respond with ONLY JSON: {"max_particle_size_mm": <8|16|32>, "grading": "<coarse|medium|fine>"}'''

def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def call_frontier_zs(image_b64, prompt):
    resp = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT, max_tokens=256,
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{image_b64}'}},
            {'type': 'text', 'text': prompt},
        ]}],
    )
    return resp.choices[0].message.content

def call_frontier_fs(ref_b64, image_b64, ref_prompt, query_prompt):
    resp = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT, max_tokens=256,
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{ref_b64}'}},
            {'type': 'text', 'text': ref_prompt},
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{image_b64}'}},
            {'type': 'text', 'text': query_prompt},
        ]}],
    )
    return resp.choices[0].message.content

print(f'Azure OpenAI: {AZURE_OPENAI_ENDPOINT}')
print(f'Deployment: {AZURE_OPENAI_DEPLOYMENT}')
print('Client ready.')


### Frontier Quick Test (1 per class, zero-shot)

In [ ]:
import time as _time

all_classes = sorted(set(e['class'] for e in manifest))
quick = [next(e for e in manifest if e['class'] == cls) for cls in all_classes]

print(f'Frontier zero-shot quick test: {len(quick)} images')
print(f'{"Class":<6} {"GT Size":>8} {"GT Grade":>10} {"Pred Size":>10} {"Pred Grade":>12} {"Size":>5} {"Grade":>6}')
print('-' * 60)

for entry in quick:
    img_path = os.path.join(TEST_DIR, entry['image'])
    img_b64 = encode_image(img_path)
    try:
        raw = call_frontier_zs(img_b64, FRONTIER_PROMPT_ZS)
        parsed = parse_response(raw)
    except Exception as e:
        print(f'{entry["class"]:<6} ERROR: {e}')
        continue
    gt_s = entry['max_particle_size_mm']; gt_g = entry['grading']
    if parsed:
        ps = parsed.get('max_particle_size_mm'); pg = parsed.get('grading', '').lower()
        if isinstance(ps, str): ps = int(ps) if ps.isdigit() else ps
        sv = '✓' if ps == gt_s else '✗'; gv = '✓' if pg == gt_g else '✗'
        print(f'{entry["class"]:<6} {gt_s:>8} {gt_g:>10} {str(ps):>10} {pg:>12} {sv:>5} {gv:>6}')
    else:
        print(f'{entry["class"]:<6} {gt_s:>8} {gt_g:>10} {"PARSE FAIL":>10} {"":>12}')
    _time.sleep(0.5)


### Frontier Full Benchmark — Zero-Shot (108 images)
Sends original full-res images to the frontier model API. ~2-5 min depending on rate limits.

In [ ]:
import time as _time

print(f'Running frontier zero-shot benchmark on {len(manifest)} images...')
fr_zs_results = []
fr_zs_size = 0; fr_zs_grading = 0; fr_zs_json = 0; fr_zs_time = 0

for i, entry in enumerate(manifest):
    img_path = os.path.join(TEST_DIR, entry['image'])
    if not os.path.exists(img_path): continue
    img_b64 = encode_image(img_path)
    t = _time.time()
    try:
        raw = call_frontier_zs(img_b64, FRONTIER_PROMPT_ZS)
    except Exception as e:
        raw = f'ERROR: {e}'
    elapsed = _time.time() - t; fr_zs_time += elapsed
    parsed = parse_response(raw)
    gt_s = entry['max_particle_size_mm']; gt_g = entry['grading']
    s_ok = False; g_ok = False
    if parsed:
        fr_zs_json += 1
        ps = parsed.get('max_particle_size_mm')
        if isinstance(ps, str): ps = int(ps) if ps.isdigit() else None
        if ps == gt_s: s_ok = True; fr_zs_size += 1
        if parsed.get('grading', '').lower() == gt_g: g_ok = True; fr_zs_grading += 1
    fr_zs_results.append({'image': entry['image'], 'class': entry['class'],
        'gt_size': gt_s, 'gt_grading': gt_g, 'predicted': parsed, 'raw': raw,
        'size_correct': s_ok, 'grading_correct': g_ok, 'valid_json': parsed is not None, 'time_s': round(elapsed, 2)})
    if (i+1) % 20 == 0:
        n = i+1
        print(f'  [{n}/{len(manifest)}] Size: {fr_zs_size}/{n} ({fr_zs_size/n*100:.0f}%) | '
              f'Grading: {fr_zs_grading}/{n} ({fr_zs_grading/n*100:.0f}%) | JSON: {fr_zs_json}/{n}')
    _time.sleep(0.3)  # rate limiting

print(f'\nDone. {len(fr_zs_results)} images in {fr_zs_time:.0f}s')


### Frontier Full Benchmark — Few-Shot (108 images)
Sends reference chart + test image to the frontier model.

In [ ]:
# Encode reference image
ref_b64 = encode_image(REF_IMAGE_PATH)
print(f'Reference image encoded.')

print(f'Running frontier few-shot benchmark on {len(manifest)} images...')
fr_fs_results = []
fr_fs_size = 0; fr_fs_grading = 0; fr_fs_json = 0; fr_fs_time = 0

for i, entry in enumerate(manifest):
    img_path = os.path.join(TEST_DIR, entry['image'])
    if not os.path.exists(img_path): continue
    img_b64 = encode_image(img_path)
    t = _time.time()
    try:
        raw = call_frontier_fs(ref_b64, img_b64, FRONTIER_PROMPT_FS_REF, FRONTIER_PROMPT_FS_QUERY)
    except Exception as e:
        raw = f'ERROR: {e}'
    elapsed = _time.time() - t; fr_fs_time += elapsed
    parsed = parse_response(raw)
    gt_s = entry['max_particle_size_mm']; gt_g = entry['grading']
    s_ok = False; g_ok = False
    if parsed:
        fr_fs_json += 1
        ps = parsed.get('max_particle_size_mm')
        if isinstance(ps, str): ps = int(ps) if ps.isdigit() else None
        if ps == gt_s: s_ok = True; fr_fs_size += 1
        if parsed.get('grading', '').lower() == gt_g: g_ok = True; fr_fs_grading += 1
    fr_fs_results.append({'image': entry['image'], 'class': entry['class'],
        'gt_size': gt_s, 'gt_grading': gt_g, 'predicted': parsed, 'raw': raw,
        'size_correct': s_ok, 'grading_correct': g_ok, 'valid_json': parsed is not None, 'time_s': round(elapsed, 2)})
    if (i+1) % 20 == 0:
        n = i+1
        print(f'  [{n}/{len(manifest)}] Size: {fr_fs_size}/{n} ({fr_fs_size/n*100:.0f}%) | '
              f'Grading: {fr_fs_grading}/{n} ({fr_fs_grading/n*100:.0f}%) | JSON: {fr_fs_json}/{n}')
    _time.sleep(0.3)

print(f'\nDone. {len(fr_fs_results)} images in {fr_fs_time:.0f}s')


### Frontier vs Qwen2.5-VL-3B Comparison

In [ ]:
def make_summary(label, results, c_size, c_grading, v_json, t_time):
    n = len(results)
    both = sum(1 for r in results if r['size_correct'] and r['grading_correct'])
    return {'label': label, 'n': n, 'json': round(v_json/n*100,1), 'size': round(c_size/n*100,1),
            'grading': round(c_grading/n*100,1), 'both': round(both/n*100,1), 'time': round(t_time/n,2)}

fr_zs_s = make_summary('Frontier ZS', fr_zs_results, fr_zs_size, fr_zs_grading, fr_zs_json, fr_zs_time)
fr_fs_s = make_summary('Frontier FS', fr_fs_results, fr_fs_size, fr_fs_grading, fr_fs_json, fr_fs_time)

print(f'{"Method":<28} {"JSON":>6} {"Size":>7} {"Grade":>7} {"Both":>7} {"Time":>6}')
print('-' * 63)

# Load Qwen baselines
for mode in ['zero-shot', 'few-shot']:
    path = f'benchmark_results_{mode}.json'
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        print(f'Qwen2.5-VL-3B ({mode}){" "*(14-len(mode))} {d["json_validity_pct"]:>5.0f}% {d["size_accuracy_pct"]:>6.1f}% '
              f'{d["grading_accuracy_pct"]:>6.1f}% {d["both_correct_pct"]:>6.1f}% {d["avg_inference_time_s"]:>5.1f}s')

model_name = AZURE_OPENAI_DEPLOYMENT
print(f'{model_name} (zero-shot){" "*(16-len(model_name))} {fr_zs_s["json"]:>5.0f}% {fr_zs_s["size"]:>6.1f}% '
      f'{fr_zs_s["grading"]:>6.1f}% {fr_zs_s["both"]:>6.1f}% {fr_zs_s["time"]:>5.1f}s')
print(f'{model_name} (few-shot){" "*(17-len(model_name))} {fr_fs_s["json"]:>5.0f}% {fr_fs_s["size"]:>6.1f}% '
      f'{fr_fs_s["grading"]:>6.1f}% {fr_fs_s["both"]:>6.1f}% {fr_fs_s["time"]:>5.1f}s')


### Frontier Per-Class Breakdown

In [ ]:
for label, results in [('Frontier ZS', fr_zs_results), ('Frontier FS', fr_fs_results)]:
    print(f'\n{label}:')
    print(f'{"Class":<6} {"Size":>6} {"Grade":>6} {"Both":>6}')
    print('-' * 28)
    by_class = defaultdict(list)
    for r in results: by_class[r['class']].append(r)
    for cls in sorted(by_class):
        cr = by_class[cls]
        sc = sum(1 for r in cr if r['size_correct'])
        gc = sum(1 for r in cr if r['grading_correct'])
        bc = sum(1 for r in cr if r['size_correct'] and r['grading_correct'])
        print(f'{cls:<6} {sc:>3}/{len(cr)} {gc:>3}/{len(cr)} {bc:>3}/{len(cr)}')


### Save Frontier Results

In [ ]:
model_name = AZURE_OPENAI_DEPLOYMENT

for label, s, results in [
    ('frontier-zero-shot', fr_zs_s, fr_zs_results),
    ('frontier-few-shot', fr_fs_s, fr_fs_results),
]:
    fname = f'benchmark_results_{label}.json'
    with open(fname, 'w') as f:
        json.dump({
            'model': model_name, 'mode': label, 'phase': 'frontier_baseline',
            'total_images': s['n'], 'json_validity_pct': s['json'],
            'size_accuracy_pct': s['size'], 'grading_accuracy_pct': s['grading'],
            'both_correct_pct': s['both'], 'avg_inference_time_s': s['time'],
            'results': results,
        }, f, indent=2)
    print(f'Saved {fname}')

print(f'\nIf frontier model scores well, proceed to Task 4 using it as the teacher.')
print(f'If frontier model also fails, the task may need different prompting or more training data.')
